# Imports


In [4]:
# !pip install -q opencv-python scikit-image pandas tqdm

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import xml.etree.ElementTree as ET
from skimage import draw
from scipy import ndimage
from tqdm.auto import tqdm
import zipfile


import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from skimage.measure import label as cc_label

import torchvision.transforms as T
from scipy import ndimage


if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
    
print("DEVICE:", DEVICE)

DEVICE: mps


In [6]:
BASE = Path("./data")
TRAIN_DIR = BASE / "train"
VAL_DIR  = BASE / "val"
TEST_DIR  = BASE / "test_final"

CLASSES = ["Epithelial","Lymphocyte","Neutrophil","Macrophage"]
IMG_SIZE = 320
BATCH_SIZE = 4
EPOCHS = 5
LR = 1e-3


In [3]:
def parse_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    annotations = []

    for annotation in root.findall('Annotation'):
        cell_type = None
        attributes = annotation.find('Attributes')
        if attributes is not None:
            attribute = attributes.find('Attribute')
            if attribute is not None:
                cell_type = attribute.get('Name')

        if not cell_type:
            continue

        regions_element = annotation.find('Regions')
        if regions_element is not None:
            regions = regions_element.findall('Region')

            for region in regions:
                vertices = []
                vertices_element = region.find('Vertices')

                if vertices_element is not None:
                    for vertex in vertices_element.findall('Vertex'):
                        x = float(vertex.get('X'))
                        y = float(vertex.get('Y'))
                        vertices.append((x, y))

                if vertices:
                    annotations.append({
                        'cell_type': cell_type,
                        'vertices': vertices
                    })

    return annotations

def generate_masks(image_shape, annotations):
    height, width = image_shape[:2]

    masks = {
        'Epithelial': np.zeros((height, width), dtype=np.uint16),
        'Lymphocyte': np.zeros((height, width), dtype=np.uint16),
        'Neutrophil': np.zeros((height, width), dtype=np.uint16),
        'Macrophage': np.zeros((height, width), dtype=np.uint16)
    }

    instance_id = {
        'Epithelial': 1,
        'Lymphocyte': 1,
        'Neutrophil': 1,
        'Macrophage': 1
    }

    for ann in annotations:
        cell_type = ann['cell_type']
        if cell_type in masks:
            vertices = np.array(ann['vertices'])
            rr, cc = draw.polygon(vertices[:, 1], vertices[:, 0], shape=(height, width))
            masks[cell_type][rr, cc] = instance_id[cell_type]
            instance_id[cell_type] += 1

    return masks

def rle_encode(mask):
    pixels = mask.flatten(order="F").astype(np.int32)
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1

    rle_parts = []
    for i in range(len(runs)-1):
        start = runs[i]
        end = runs[i+1] if i+1 < len(runs) else len(pixels)-1
        length = end - start
        value = pixels[start]

        if value > 0:
            rle_parts.extend([value, start, length])

    return ' '.join(str(x) for x in rle_parts) if rle_parts else ""

def rle_decode(mask_rle, shape):
    if isinstance(mask_rle, float) and np.isnan(mask_rle):
        return np.zeros(shape, dtype=np.uint16)

    if not mask_rle or mask_rle.strip() == '':
        return np.zeros(shape, dtype=np.uint16)

    s = str(mask_rle).split()
    if len(s) == 0 or len(s) % 3 != 0:
        return np.zeros(shape, dtype=np.uint16)

    img = np.zeros(shape[0] * shape[1], dtype=np.uint16)

    for i in range(0, len(s), 3):
        value = int(s[i])
        start = int(s[i+1]) - 1
        length = int(s[i+2])
        img[start:start+length] = value

    return img.reshape(shape, order='F')


In [ ]:
def display_image_annotation(annotated_image):
    image = annotated_image['image'].cpu().permute(1, 2, 0)
    masks = annotated_image['mask'].cpu()

    # print(masks.shape)
    plt.figure(figsize=(10,5))

    plt.subplot(161)
    plt.imshow(image)
    plt.axis('off') # Turn off axis labels
    plt.title(annotated_image['image_id'])

    combined_mask = np.zeros_like(masks[0])

    for i in range(0,4):

        # print(masks[i])

        as_np = masks[i].detach().numpy().astype(np.int32)

        combined_mask = combined_mask + (as_np* (i + 1))

        # unique_values = np.unique(as_np)
        # Print the unique values
        # print(unique_values)
        # print(as_np.shape)

        plt.subplot(162+i)
        plt.imshow(as_np)
        plt.axis('off') # Turn off axis labels
        plt.title(CLASSES[i] + "\nmask" )

    plt.subplot(166)
    plt.imshow(combined_mask)
    plt.axis('off') # Turn off axis labels
    plt.title("Combined\nmask")

    plt.show()

In [ ]:
train_ground_truth = pd.read_csv(str(TRAIN_DIR)+ "/train_ground_truth.csv")

train_ground_truth.head()

In [ ]:
val_ground_truth = pd.read_csv(str(VAL_DIR)+ "/val_truth.csv").fillna(0)
val_ground_truth.head()

In [ ]:
class CellDataset(Dataset):
    def __init__(self, data_dir, image_size=256):
        self.data_dir = Path(data_dir)
        self.image_size = image_size
        self.xml_files = list(self.data_dir.glob("*.xml"))

    def __len__(self):
        return len(self.xml_files)

    def __getitem__(self, idx):
        xml_path = self.xml_files[idx]
        img_path = xml_path.with_suffix('.tif')

        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        original_shape = image.shape[:2]
        image = cv2.resize(image, (self.image_size, self.image_size))

        annotations = parse_xml(xml_path)
        masks = generate_masks(original_shape, annotations)

        mask = np.zeros((4, self.image_size, self.image_size), dtype=np.float32)
        for i, cell_type in enumerate(CLASSES):
            cell_mask = masks[cell_type]
            binary_mask = (cell_mask > 0).astype(np.float32)
            binary_mask = cv2.resize(binary_mask, (self.image_size, self.image_size))
            mask[i] = binary_mask

        image = image.astype(np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))

        return {
            'image': torch.from_numpy(image),
            'mask': torch.from_numpy(mask),
            'image_id': xml_path.stem,
            'original_shape': original_shape
        }


### Model


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, is_enc = False):
        super().__init__()

        if is_enc == True:
            self.conv = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.LeakyReLU(inplace=True),
                nn.Conv2d(out_channels, out_channels, 3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.LeakyReLU(inplace=True)
            )
        else:
            self.conv = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_channels, out_channels, 3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True)
            )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=4):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, 64, is_enc = True)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(64, 128, is_enc = True)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(128, 256, is_enc = True)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        self.out = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool1(enc1))
        enc3 = self.enc3(self.pool2(enc2))

        bottleneck = self.bottleneck(self.pool3(enc3))

        dec3 = self.up3(bottleneck)
        dec3 = torch.cat([dec3, enc3], dim=1)
        dec3 = self.dec3(dec3)

        dec2 = self.up2(dec3)
        dec2 = torch.cat([dec2, enc2], dim=1)
        dec2 = self.dec2(dec2)

        dec1 = self.up1(dec2)
        dec1 = torch.cat([dec1, enc1], dim=1)
        dec1 = self.dec1(dec1)

        return torch.sigmoid(self.out(dec1))

def dice_loss(pred, target, smooth=1e-5):
    pred = pred.contiguous()
    target = target.contiguous()

    intersection = (pred * target).sum(dim=2).sum(dim=2)
    loss = (1 - ((2. * intersection + smooth) /
                 (pred.sum(dim=2).sum(dim=2) + target.sum(dim=2).sum(dim=2) + smooth)))

    return loss.mean()

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler

def train_model(model, train_loader, num_epochs, device):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.5, total_iters=30)

    print(f"\nTraining on {device}")

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0

        for batch_idx, batch in enumerate(train_loader):
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)

            optimizer.zero_grad()
            outputs = model(images)

            class_weights = torch.tensor([1,1,10,10]).to(device)
            loss = (dice_loss(outputs, masks) * class_weights).mean()


            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            if batch_idx % 5 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}] Batch [{batch_idx}/{len(train_loader)}] Loss: {loss.item():.4f}")

        scheduler.step()

        avg_loss = epoch_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{num_epochs}] Avg Loss: {avg_loss:.4f}")

    return model

In [ ]:
def predict_test_set(model, test_dir, device, threshold=0.5):
    model = model.to(device)
    model.eval()

    test_tif_files = list(Path(test_dir).glob("*.tif"))
    submission_rows = []

    print(f"\nPredicting on {len(test_tif_files)} test images...")

    with torch.no_grad():
        for tif_file in tqdm(test_tif_files):
            image = cv2.imread(str(tif_file))
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            original_shape = image_rgb.shape[:2]

            image_resized = cv2.resize(image_rgb, (IMG_SIZE, IMG_SIZE))
            image_norm = image_resized.astype(np.float32) / 255.0
            image_tensor = torch.from_numpy(np.transpose(image_norm, (2, 0, 1))).unsqueeze(0).to(device)

            output = model(image_tensor)
            output = output.squeeze(0).cpu().numpy()

            instance_masks = {}
            for i, cell_type in enumerate(CLASSES):
                mask = output[i]
                mask = cv2.resize(mask, (original_shape[1], original_shape[0]))
                binary_mask = (mask > threshold).astype(np.uint8)

                labeled_mask, _ = ndimage.label(binary_mask)
                instance_masks[cell_type] = labeled_mask.astype(np.uint16)

            image_id = tif_file.stem
            row = {'image_id': image_id}

            for cell_type in CLASSES:
                mask = instance_masks[cell_type]
                row[cell_type] = rle_encode(mask) if np.max(mask) > 0 else ""

            submission_rows.append(row)

    return pd.DataFrame(submission_rows)

In [ ]:
def map_to_val(x, val):
    if x > 0.1:  # Example condition
        return val
    else:
        return 0

def display_predicted_masks(submission, train_val_or_test):

    # print(submission.head())


    for index, row in submission.iterrows():
        print(row)

        image_id = row["image_id"]

        if train_val_or_test == "train":
            s = str(TRAIN_DIR) + "\\" + str(image_id) + ".tif"
        elif train_val_or_test == "val":
            s = str(VAL_DIR) + "\\" + str(image_id) + ".tif"
        else:
            s = str(TEST_DIR) + "\\" + str(image_id) + ".tif"

        image = plt.imread(s)

        plt.figure(figsize=(10,5))

        plt.subplot(161)
        plt.imshow(image)
        plt.axis('off') # Turn off axis labels
        plt.title(image_id)


        # print(type(row))
        combined_mask = np.zeros(image.shape[:2])
        # print(combined_mask.shape)

        for i in range(1,5):

            decoded_mask = rle_decode(row[i], image.shape[:2])

            vectorized_map = np.vectorize(map_to_val)

            mapped_mask = vectorized_map(decoded_mask, i)

            combined_mask = combined_mask + mapped_mask

            plt.subplot(161+i)
            plt.imshow(mapped_mask)
            plt.axis('off') # Turn off axis labels
            plt.title(CLASSES[i-1] + "\nmask" )

        plt.subplot(166)
        plt.imshow(combined_mask)
        plt.axis('off') # Turn off axis labels
        plt.title("Combined\nmask")

        plt.show()

# start = 1
# stop = 10

# # display_predicted_masks(train_ground_truth.iloc[start: stop+1])
# display_predicted_masks(val_pred_df.iloc[start: stop+1], train_val_or_test="val")

### Train


In [ ]:
print("Creating dataset...")

train_dataset = CellDataset(TRAIN_DIR, image_size=IMG_SIZE)
validation_dataset = CellDataset(VAL_DIR, image_size=IMG_SIZE)

# train_len = int(len(full_dataset) * 0.8)
# val_len = len(full_dataset) - train_len

# train_dataset, val_dataset = random_split(full_dataset, [train_len, val_len],generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
print(f"Train samples: {len(train_dataset)}")

validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Validate samples: {len(validation_dataset)}")


# Accessing elements from the subsets
# print(f"First element of training dataset: {train_dataset[0]}")
# print(f"First element of validation dataset: {val_dataset[0]}")

print("\nCreating model...")
model = UNet(in_channels=3, out_channels=4)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
print("\nTraining...")
model = train_model(model, train_loader, num_epochs = EPOCHS,device = DEVICE)

### Validation


In [ ]:
print("\nGenerating validation...")
validation_df = predict_test_set(model, VAL_DIR, DEVICE, threshold=.7)

validation_df.to_csv('validation.csv', index=False)
print(f"\n✅ validation saved: validation.csv")
print(f"Shape: {validation_df.shape}")
print("\nSample rows:")
print(validation_df.head())

display_predicted_masks(validation_df[:5],train_val_or_test="val")

### Test Set Predictions


In [ ]:
print("\nGenerating predictions...")
submission_df = predict_test_set(model, TEST_DIR, DEVICE, threshold=.7)
# submission_df = submission_df.fillna(0)

submission_df.to_csv('submission_v3.csv', index=False)
print(f"\n✅ Submission saved: submission_v3.csv")
print(f"Shape: {submission_df.shape}")
print("\nSample rows:")
print(submission_df.head())

# display_predicted_masks(submission_df[:5],train_val_or_test="test")

In [ ]:
import pandas as pd

submission = pd.read_csv('./submission_v3.csv')

# Truncate RLE columns for display
submission_display = submission.copy()
for col in ['Epithelial', 'Lymphocyte', 'Neutrophil', 'Macrophage']:
    submission_display[col] = submission_display[col].astype(str).str[:40] + '...'

submission_display.head()

display_predicted_masks(submission[:5],train_val_or_test="test")

## Attempting wPQ Evaluation


In [ ]:
from skimage.metrics import adapted_rand_error
from skimage.measure import label, regionprops

In [ ]:
# ============================================================
# Weighted Panoptic Quality (wPQ) Evaluation
# ============================================================

def pq_single_class(gt_mask, pred_mask, iou_thresh=0.5):
    """
    Computes Panoptic Quality (PQ) for one class.
    Args:
        gt_mask (np.ndarray): ground truth instance mask (uint16)
        pred_mask (np.ndarray): predicted instance mask (uint16)
        iou_thresh (float): IoU threshold for matching
    Returns:
        PQ, SQ, RQ, TP, FP, FN
    """
    gt_ids = np.unique(gt_mask)
    pred_ids = np.unique(pred_mask)
    gt_ids = gt_ids[gt_ids != 0]
    pred_ids = pred_ids[pred_ids != 0]

    if len(gt_ids) == 0 and len(pred_ids) == 0:
        return 1.0, 1.0, 1.0, 0, 0, 0
    if len(gt_ids) == 0:
        return 0, 0, 0, 0, len(pred_ids), 0
    if len(pred_ids) == 0:
        return 0, 0, 0, 0, 0, len(gt_ids)

    iou_matrix = np.zeros((len(gt_ids), len(pred_ids)), dtype=np.float32)
    for i, gid in enumerate(gt_ids):
        gmask = gt_mask == gid
        for j, pid in enumerate(pred_ids):
            pmask = pred_mask == pid
            intersection = np.logical_and(gmask, pmask).sum()
            union = np.logical_or(gmask, pmask).sum()
            iou_matrix[i, j] = intersection / union if union > 0 else 0.0

    matched_gt = set()
    matched_pred = set()
    matched_ious = []

    # Greedy matching
    while True:
        max_iou = iou_matrix.max()
        if max_iou < iou_thresh:
            break
        i, j = np.unravel_index(np.argmax(iou_matrix), iou_matrix.shape)
        matched_gt.add(gt_ids[i])
        matched_pred.add(pred_ids[j])
        matched_ious.append(max_iou)
        iou_matrix[i, :] = -1
        iou_matrix[:, j] = -1

    TP = len(matched_ious)
    FP = len(pred_ids) - TP
    FN = len(gt_ids) - TP

    if TP + 0.5 * (FP + FN) == 0:
        RQ = 0
    else:
        RQ = TP / (TP + 0.5 * (FP + FN))
    SQ = np.mean(matched_ious) if matched_ious else 0
    PQ = SQ * RQ

    return PQ, SQ, RQ, TP, FP, FN


def weighted_pq_per_image(gt_masks_dict, pred_masks_dict, class_weights=None, iou_thresh=0.5):
    """
    Computes Weighted PQ across all 4 classes for one image.

    Args:
        gt_masks_dict (dict): {'Epithelial': mask, 'Lymphocyte': mask, ...}
        pred_masks_dict (dict): same keys as above
        class_weights (dict): optional weights per class
    """
    if class_weights is None:
        class_weights = {'Epithelial': 1, 'Lymphocyte': 1, 'Neutrophil': 10, 'Macrophage': 10}

    results = {}
    total_weight = 0
    weighted_sum = 0

    for cls in CLASSES:
        pq, sq, rq, TP, FP, FN = pq_single_class(gt_masks_dict[cls], pred_masks_dict[cls], iou_thresh)
        results[cls] = {'PQ': pq, 'SQ': sq, 'RQ': rq, 'TP': TP, 'FP': FP, 'FN': FN}
        weighted_sum += pq * class_weights[cls]
        total_weight += class_weights[cls]

    results['WeightedPQ'] = weighted_sum / total_weight if total_weight > 0 else 0
    return results


In [ ]:
print("\nEvaluating Weighted PQ on validation set...")

all_results = []
for idx, row in tqdm(validation_df.iterrows(), total=len(validation_df)):
    image_id = row['image_id']

    # Ground truth RLEs for this image
    gt_row = val_ground_truth[val_ground_truth['image_id'] == image_id]
    if gt_row.empty:
        continue

    gt_masks = {}
    pred_masks = {}

    for cls in CLASSES:
        gt_rle = gt_row.iloc[0][cls] if cls in gt_row.columns else ""
        pred_rle = row[cls] if cls in row else ""
        gt_masks[cls] = rle_decode(gt_rle, (IMG_SIZE, IMG_SIZE))
        pred_masks[cls] = rle_decode(pred_rle, (IMG_SIZE, IMG_SIZE))

    results = weighted_pq_per_image(gt_masks, pred_masks)
    results['image_id'] = image_id
    all_results.append(results)

# Convert results to DataFrame
pq_df = pd.DataFrame(all_results)
print("\nSample PQ results per image:")
print(pq_df.head())

mean_weighted_pq = pq_df['WeightedPQ'].mean()
print(f"\nMean Weighted PQ across validation set: {mean_weighted_pq:.4f}")


#### The model is TERRIBLE as is!!!
